# Mixed Init-Route Dataset Generation

Строит **смешанный** seeded-route датасет из `raw_graphs_1000.pkl`: каждому
графу назначается набор маршрутов, сгенерированный одним из нескольких методов.
По аналогии с `nx_heuristic_dataset_generation.ipynb` и тем, как строятся
init-маршруты в `benchmark_init_visualize.ipynb` / `eval_lib.baselines`.

## Типы генерации

| Тип | Метод | Длины | Особенность |
|---|---|---|---|
| `heuristic`  | NX max-length heuristic | min=max=12 | фиксированная длина (как было) |
| `lc_a0.0`    | LC-10 construction      | ≤12 | alpha=0  → route-only |
| `lc_a0.5`    | LC-10 construction      | ≤12 | alpha=0.5 → balanced |
| `lc_a1.0`    | LC-10 construction      | ≤12 | alpha=1  → demand-only |
| `rpc`        | Random path combiner    | 5..12 | min=5, max=12 |

У всех `max_route_len = 12`. `alpha` задаёт баланс cost-весов
(`demand_time_weight = alpha`, `route_time_weight = 1 - alpha`; connectivity
отключён через `DISABLED_COST_COMPONENTS`).

**Сколько графов каждого типа** генерить — одна переменная `N_PER_TYPE`
(по умолчанию 1 для теста). Итого графов = `N_PER_TYPE * len(GENERATION_TYPES)`,
каждый берётся из своего сырого графа.

## Что сохраняется

Новая папка `datasets/mixed_init_routes_10r_len12/`:
- `graph_XXXX/lc_<type>_graph_XXXX_routes_routes.pkl` — маршруты (формат как у
  `nx_heuristic_results`, читается `load_raw_graphs_and_lc_routes`);
- `graph_XXXX/metrics.txt`;
- `raw_graphs_subset.pkl` — подмножество сырых графов, выровненное по индексам с
  `graph_XXXX/` (чтобы датасет грузился целиком);
- `all_results_summary.csv`.

In [1]:
import os as _os, sys as _sys
_NB_DIR = _os.path.abspath(".")
if _NB_DIR not in _sys.path:
    _sys.path.insert(0, _NB_DIR)

import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from IPython.display import display

import eval_lib
from eval_lib import *  # noqa: F401,F403
from eval_lib import plots as _plots

from connectpt.routes_generator.nx_heuristic import build_nx_heuristic_routes
from connectpt.routes_generator.citygraph_dataset import STOP_KEY
from connectpt.routes_generator.transit_time_estimator import RouteGenBatchState
from connectpt.routes_generator.torch_utils import dump_routes
from connectpt.routes_generator.eval_route_generator import eval_model
from connectpt.routes_generator.improvement_learning import (
    load_raw_graphs_and_lc_routes)
import connectpt.routes_generator.utils as lrnu
from connectpt.routes_generator.utils import get_eval_cfg

pd.set_option("display.max_columns", None)
print("eval_lib + connectpt loaded")

eval_lib + connectpt loaded


## Конфигурация

In [2]:
# --- Главный knob: сколько графов на КАЖДЫЙ тип (1 для теста) ---
N_PER_TYPE = 1

# --- Контракт маршрутов ---
N_ROUTES_OUT  = 10     # маршрутов на граф ("LC-10")
MAX_ROUTE_LEN = 12     # общий max для всех типов
LC_N_SAMPLES  = 10     # сэмплов конструктора LC ("LC-10")
RPC_N_SAMPLES = 1      # RPC: один сэмпл (как в benchmark)

# Heuristic: min=max (фикс. длина). NX-эвристика всё равно выдаёт ровно max_len;
# min влияет только на пул seed-путей. Если кандидатов мало и генерация падает
# на конкретном графе -- авто-fallback на min_len=2 (см. generate_routes_for_type).
HEURISTIC_MIN_LEN = MAX_ROUTE_LEN
RPC_MIN_LEN       = 5
LC_MIN_LEN        = 2   # конструктор LC строит маршруты переменной длины <=max

DATASET_SEED = 0

NEW_DATASET_DIR = DATASETS_DIR / "mixed_init_routes_10r_len12"

GENERATION_TYPES = [
    {"key": "heuristic", "method": "nx_heuristic",
     "label": "NX heuristic (min=max=12)",
     "min_len": HEURISTIC_MIN_LEN, "max_len": MAX_ROUTE_LEN,
     "alpha": None, "n_samples": None},
    {"key": "lc_a0.0", "method": "lc",
     "label": "LC-10 (alpha=0, route-only)",
     "min_len": LC_MIN_LEN, "max_len": MAX_ROUTE_LEN,
     "alpha": 0.0, "n_samples": LC_N_SAMPLES},
    {"key": "lc_a0.5", "method": "lc",
     "label": "LC-10 (alpha=0.5, balanced)",
     "min_len": LC_MIN_LEN, "max_len": MAX_ROUTE_LEN,
     "alpha": 0.5, "n_samples": LC_N_SAMPLES},
    {"key": "lc_a1.0", "method": "lc",
     "label": "LC-10 (alpha=1, demand-only)",
     "min_len": LC_MIN_LEN, "max_len": MAX_ROUTE_LEN,
     "alpha": 1.0, "n_samples": LC_N_SAMPLES},
    {"key": "rpc", "method": "rpc",
     "label": "RPC (min=5, max=12)",
     "min_len": RPC_MIN_LEN, "max_len": MAX_ROUTE_LEN,
     "alpha": None, "n_samples": RPC_N_SAMPLES},
]

print(f"N_PER_TYPE:      {N_PER_TYPE}")
print(f"Types:           {[t['key'] for t in GENERATION_TYPES]}")
print(f"Total graphs:    {N_PER_TYPE * len(GENERATION_TYPES)}")
print(f"n_routes/graph:  {N_ROUTES_OUT}")
print(f"max_route_len:   {MAX_ROUTE_LEN}")
print(f"Disabled cost:   {DISABLED_COST_COMPONENTS}")
print(f"Output dir:      {NEW_DATASET_DIR}")

N_PER_TYPE:      1
Types:           ['heuristic', 'lc_a0.0', 'lc_a0.5', 'lc_a1.0', 'rpc']
Total graphs:    5
n_routes/graph:  10
max_route_len:   12
Disabled cost:   ['connectivity']
Output dir:      D:\PythonProjects\connectpt\datasets\mixed_init_routes_10r_len12


## Вспомогательные функции

- общий `EVAL_COST_OBJ` для сопоставимых метрик (cost/ATT/RTT/d_un) по всем типам;
- `run_construction_on_graph` — гоняет LC/RPC модель напрямую на сыром графе
  (через 1-граф DataLoader, без ре-трансформации тензоров);
- `generate_routes_for_type` — диспетчер по методам;
- `_to_fixed_route_tensor` — приводит результат к `[n_routes, max_len]` с -1-паддингом.

In [3]:
# Общий cost-objective: метрики всех типов считаются под одним балансом
# (по умолчанию из eval_model_mumford), чтобы строки таблицы были сравнимы.
_eval_cfg = get_eval_cfg(
    str(CFG_DIR), "eval_model_mumford",
    {"dataset_name": "tensor", "n_routes": N_ROUTES_OUT,
     "min_route_len": 1, "max_route_len": MAX_ROUTE_LEN,
     "run_name": "mixed_init_eval"})
_eval_cfg.batch_size = 1
EVAL_DEVICE, _, _, EVAL_COST_OBJ, _ = lrnu.process_standard_experiment_cfg(
    _eval_cfg, run_name_prefix="mixed_init_eval_", weights_required=False)
EVAL_COST_OBJ.ignore_stops_oob = True
print(f"Eval device: {EVAL_DEVICE}")


def _to_fixed_route_tensor(routes, n_routes, max_route_len):
    """Normalize any route output to [n_routes, max_route_len] with -1 padding."""
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < n_routes:
        pad = torch.full((n_routes - t.shape[0], t.shape[1]), -1, dtype=t.dtype)
        t = torch.cat([t, pad], dim=0)
    elif t.shape[0] > n_routes:
        t = t[:n_routes]
    if t.shape[1] < max_route_len:
        pad = torch.full((t.shape[0], max_route_len - t.shape[1]), -1, dtype=t.dtype)
        t = torch.cat([t, pad], dim=1)
    elif t.shape[1] > max_route_len:
        if (t[:, max_route_len:] >= 0).any():
            raise ValueError("route exceeds max_route_len with real stops")
        t = t[:, :max_route_len]
    return t


def run_construction_on_graph(cfg, graph, n_samples):
    """Run an LC/RPC construction model once on a single raw CityGraphData."""
    dataloader = DataLoader([graph.clone()], batch_size=1)
    device, _rn, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
        cfg, run_name_prefix="mixed_init_gen_", weights_required=True)
    if hasattr(model, "clear_step_counts_log"):
        model.clear_step_counts_log()
    _, _unserved, _metrics, routes = eval_model(
        model, dataloader, cfg.eval, cost_obj,
        n_samples=int(n_samples), return_routes=True,
        silent=True, device=device)
    return as_route_tensor(routes)


def generate_routes_for_type(tcfg, graph, seed):
    """Generate one route set for `graph` using the method in `tcfg`."""
    method = tcfg["method"]
    torch.manual_seed(seed)
    if method == "nx_heuristic":
        try:
            routes = build_nx_heuristic_routes(
                graph, num_routes=N_ROUTES_OUT,
                min_len=tcfg["min_len"], max_len=tcfg["max_len"], seed=seed)
        except ValueError as exc:
            print(f"  [heuristic] min_len={tcfg['min_len']} failed ({exc}); "
                  f"falling back to min_len=2")
            routes = build_nx_heuristic_routes(
                graph, num_routes=N_ROUTES_OUT,
                min_len=2, max_len=tcfg["max_len"], seed=seed)
    elif method == "lc":
        alpha = float(tcfg["alpha"])
        cfg = build_lc_cfg(
            run_name=f"mixed_lc_a{alpha}",
            n_routes=N_ROUTES_OUT,
            min_route_len=tcfg["min_len"], max_route_len=tcfg["max_len"],
            demand_time_weight=alpha,
            route_time_weight=1.0 - alpha,
            median_connectivity_weight=0.0)
        routes = run_construction_on_graph(cfg, graph, tcfg["n_samples"])
    elif method == "rpc":
        cfg = build_rpc_cfg(
            run_name="mixed_rpc",
            n_routes=N_ROUTES_OUT,
            min_route_len=tcfg["min_len"], max_route_len=tcfg["max_len"])
        routes = run_construction_on_graph(cfg, graph, tcfg["n_samples"])
    else:
        raise ValueError(f"unknown method {method}")
    return _to_fixed_route_tensor(routes, N_ROUTES_OUT, MAX_ROUTE_LEN)


def evaluate_route_set(graph, routes_tensor):
    """Score a route set under the shared EVAL_COST_OBJ for comparable metrics."""
    graph_batch = Batch.from_data_list([graph.clone()]).to(EVAL_DEVICE)
    rt = routes_tensor.to(EVAL_DEVICE)
    if rt.ndim == 2:
        rt = rt[None]
    state = RouteGenBatchState(
        graph_batch, EVAL_COST_OBJ, N_ROUTES_OUT, 1, MAX_ROUTE_LEN)
    state.add_new_routes(rt)
    result = EVAL_COST_OBJ(state)
    out = {}
    for key, val in result.get_metrics().items():
        if torch.is_tensor(val):
            val = val.detach().reshape(-1)[0].cpu().item()
        out[key] = float(val)
    return out


print("helpers ready")

Eval device: cpu
helpers ready


## Генерация датасета

In [4]:
raw_graphs = pd.read_pickle(RAW_GRAPHS_PATH)
print(f"Loaded {len(raw_graphs)} raw graphs from {RAW_GRAPHS_PATH}\n")

NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)

records = []
subset_graphs = []
GENERATED_ROUTES = {}   # global_idx -> route tensor [n_routes, max_len]
global_idx = 0
source_idx = 0

for tcfg in GENERATION_TYPES:
    for _ in range(N_PER_TYPE):
        if source_idx >= len(raw_graphs):
            raise RuntimeError("Ran out of raw graphs for the requested counts")
        graph = raw_graphs[source_idx]
        seed = DATASET_SEED + global_idx

        routes = generate_routes_for_type(tcfg, graph, seed)
        metrics = evaluate_route_set(graph, routes)

        graph_dir = NEW_DATASET_DIR / f"graph_{global_idx:04d}"
        graph_dir.mkdir(parents=True, exist_ok=True)
        run_name = f"lc_{tcfg['key']}_graph_{global_idx:04d}"
        dump_routes(f"{run_name}_routes", routes, out_dir=graph_dir)

        route_lens = [int((r >= 0).sum().item()) for r in routes]
        covered = len({int(n) for r in routes for n in r.tolist() if n >= 0})
        rec = {
            "global_index": global_idx,
            "source_graph_index": source_idx,
            "type_key": tcfg["key"],
            "method": tcfg["method"],
            "label": tcfg["label"],
            "alpha": tcfg["alpha"],
            "n_samples": tcfg["n_samples"],
            "min_len": tcfg["min_len"],
            "max_len": tcfg["max_len"],
            "n_routes": N_ROUTES_OUT,
            "n_nodes": int(graph[STOP_KEY].num_nodes),
            "route_len_min": int(min(route_lens)),
            "route_len_mean": float(np.mean(route_lens)),
            "route_len_max": int(max(route_lens)),
            "covered_nodes": covered,
            "cost": metrics.get("cost", float("nan")),
            "ATT": metrics.get("ATT", float("nan")),
            "RTT": metrics.get("RTT", float("nan")),
            "d_un": metrics.get("$d_{un}$", float("nan")),
            "run_name": run_name,
        }
        with open(graph_dir / "metrics.txt", "w", encoding="utf-8") as fh:
            for key, val in rec.items():
                fh.write(f"{key}: {val}\n")

        records.append(rec)
        subset_graphs.append(graph)
        GENERATED_ROUTES[global_idx] = routes
        print(f"[{global_idx:04d}] {tcfg['label']:32s} "
              f"src={source_idx:<4d} cost={rec['cost']:.4f} "
              f"len[{rec['route_len_min']}-{rec['route_len_max']}] "
              f"covered={covered}/{rec['n_nodes']}")
        global_idx += 1
        source_idx += 1

# подмножество сырых графов, выровненное по индексам graph_XXXX
subset_path = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
with subset_path.open("wb") as fh:
    pickle.dump(subset_graphs, fh)

summary_df = pd.DataFrame(records)
summary_df.to_csv(NEW_DATASET_DIR / "all_results_summary.csv", index=False)

print(f"\nDataset dir:    {NEW_DATASET_DIR}")
print(f"Subset graphs:  {subset_path}  ({len(subset_graphs)} graphs)")
print(f"Summary CSV:    {NEW_DATASET_DIR / 'all_results_summary.csv'}")
display(summary_df)

Loaded 1000 raw graphs from D:\PythonProjects\connectpt\datasets\raw_graphs_1000.pkl

[0000] NX heuristic (min=max=12)        src=0    cost=2.5967 len[12-12] covered=43/50


AssertionError: input tensor has the wrong dimensionality!

## Визуализация графов разных типов

In [ ]:
n_panels = len(records)
ncols = min(n_panels, 3) if n_panels > 1 else 1
nrows = (n_panels + ncols - 1) // ncols
fig, axes = plt.subplots(
    nrows, ncols, figsize=(6.5 * ncols, 6.0 * nrows), squeeze=False)
axes = axes.flatten()

for ax, rec in zip(axes, records):
    gi = rec["global_index"]
    graph = subset_graphs[gi]
    routes = GENERATED_ROUTES[gi]
    subtitle = (
        f"src graph={rec['source_graph_index']} | cost={rec['cost']:.3f} | "
        f"len[{rec['route_len_min']}-{rec['route_len_max']}] | "
        f"covered={rec['covered_nodes']}/{rec['n_nodes']}"
    )
    _plots.plot_plain_route_set(
        ax, routes, graph,
        title=rec["label"], subtitle=subtitle,
        palette="tab20", with_overlap_curves=False)

for ax in axes[n_panels:]:
    ax.axis("off")

plt.tight_layout()
plt.show()
plt.close(fig)

## Проверка: загрузка датасета обратно

Грузим через тот же `load_raw_graphs_and_lc_routes`, что использует обучение
(на сохранённом `raw_graphs_subset.pkl`). Подтверждает что формат маршрутов и
выравнивание индексов корректны.

In [ ]:
loaded_graphs, loaded_routes = load_raw_graphs_and_lc_routes(
    subset_path, NEW_DATASET_DIR)
route_lens = (loaded_routes >= 0).sum(dim=-1)
nonempty = route_lens[route_lens > 0]

print(f"Loaded graphs:  {len(loaded_graphs)}")
print(f"Routes tensor:  {tuple(loaded_routes.shape)}")
print(f"Route length:   min={int(nonempty.min())}  max={int(nonempty.max())}")
assert len(loaded_graphs) == len(records)

print("\nPer-method route-length / metric means:")
display(
    summary_df.groupby("method")[
        ["route_len_min", "route_len_max", "route_len_mean",
         "cost", "ATT", "RTT", "d_un"]
    ].mean().round(3)
)